### Databricks notebook – Data generation code

In [0]:
%sql
Create catalog if not exists e_commerce;
create volume if not exists e_commerce.default.raw_data;

In [0]:

import random
import uuid
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

# ============================================================
# 1. CONFIGURATION
# ============================================================

random.seed()

BASE_PATH = "/Volumes/e_commerce/default/raw_data"

CUSTOMER_PATH = f"{BASE_PATH}/customer"
ORDERS_PATH = f"{BASE_PATH}/orders"
STATUS_PATH = f"{BASE_PATH}/status"

NEW_CUSTOMERS = 20
UPDATED_CUSTOMERS = 50
NEW_ORDERS = 200
STATUS_UPDATES = 100

current_time = datetime.now()
run_timestamp = current_time.strftime("%Y-%m-%d %H:%M:%S")


# ============================================================
# 2. MASTER DATA
# ============================================================

first_names = [
    "Aarav", "Vivaan", "Aditya", "Arjun", "Rahul",
    "Priya", "Ananya", "Isha", "Sneha", "Neha",
    "Rohan", "Karan", "Simran", "Meera", "Amit",
    "Kavya", "Riya", "Vikram", "Pooja", "Manish"
]

last_names = [
    "Sharma", "Verma", "Singh", "Gupta", "Negi",
    "Mehta", "Kumar", "Joshi", "Malhotra", "Reddy",
    "Patel", "Das", "Kapoor", "Yadav", "Chauhan"
]

cities = [
    "Delhi", "Mumbai", "Bangalore", "Hyderabad",
    "Pune", "Jaipur", "Lucknow", "Dehradun",
    "Chandigarh", "Noida", "Gurgaon", "Kolkata"
]

products = [
    ("P101", "Laptop", 55000),
    ("P102", "Mobile", 25000),
    ("P103", "Headphones", 2500),
    ("P104", "Smart Watch", 5000),
    ("P105", "Tablet", 18000),
    ("P106", "Keyboard", 1500),
    ("P107", "Monitor", 12000),
    ("P108", "Mouse", 800),
    ("P109", "Camera", 35000),
    ("P110", "Speaker", 3000)
]

payment_methods = [
    "UPI", "Credit Card", "Debit Card",
    "Net Banking", "COD", "Wallet"
]

# Order lifecycle
status_hierarchy = [
    "PLACED",
    "CONFIRMED",
    "PACKED",
    "SHIPPED",
    "OUT_FOR_DELIVERY",
    "DELIVERED"
]

terminal_statuses = ["DELIVERED", "CANCELLED"]


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def maybe_null(value, probability=0.05):
    return None if random.random() < probability else value


def random_date(start, end):
    days = (end - start).days
    return start + timedelta(days=random.randint(0, days))


def create_id(prefix):
    return f"{prefix}-{uuid.uuid4().hex[:12].upper()}"


def write_data(rows, path):
    if not rows:
        print(f"No new records for {path}")
        return

    schema = StructType([
        StructField(c, StringType(), True)
        for c in rows[0].keys()
    ])

    df = spark.createDataFrame(rows, schema)

    (df.write
        .format("csv")
        .mode("append")
        .option("header", True)
        .save(path))

    print(f"Inserted {len(rows)} records into {path}")


def read_existing(path):
    if not path_exists(path):
        return None

    return (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", False)
        .load(path)
    )


# ============================================================
# 4. CUSTOMER DATA
# ============================================================

existing_customer_df = read_existing(CUSTOMER_PATH)

customer_records = []

if existing_customer_df is None:

    print("FIRST RUN: Generating initial customers")

    for i in range(500):

        customer_id = create_id("CUST")
        first = random.choice(first_names)
        last = random.choice(last_names)

        customer_records.append({
            "customer_id": customer_id,
            "first_name": first,
            "last_name": last,
            "email": maybe_null(
                f"{first.lower()}.{last.lower()}@gmail.com", 0.05
            ),
            "phone": maybe_null(
                f"9{random.randint(100000000, 999999999)}", 0.04
            ),
            "city": maybe_null(random.choice(cities), 0.03),
            "gender": random.choice(["Male", "Female", "Other"]),
            "registration_date": random_date(
                datetime(2023, 1, 1), current_time
            ).strftime("%Y-%m-%d"),
            "customer_segment": random.choice(
                ["Regular", "Premium", "VIP"]
            ),
            "updated_at": run_timestamp
        })

else:

    print("INCREMENTAL RUN: Generating customer changes")

    # Get latest version of every customer
    customer_window = (
        Window.partitionBy("customer_id")
        .orderBy(F.col("updated_at").desc())
    )

    latest_customers_df = (
        existing_customer_df
        .withColumn("rn", F.row_number().over(customer_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    existing_customers = [
        row.asDict()
        for row in latest_customers_df.collect()
    ]

    # Update existing customers for SCD Type 2
    selected_customers = random.sample(
        existing_customers,
        min(UPDATED_CUSTOMERS, len(existing_customers))
    )

    for old in selected_customers:

        new_city = random.choice([
            city for city in cities if city != old["city"]
        ])

        new_segment = random.choice(
            ["Regular", "Premium", "VIP"]
        )

        customer_records.append({
            "customer_id": old["customer_id"],
            "first_name": old["first_name"],
            "last_name": old["last_name"],
            "email": f"updated.{old['customer_id'].lower()}@gmail.com",
            "phone": old["phone"],
            "city": new_city,
            "gender": old["gender"],
            "registration_date": old["registration_date"],
            "customer_segment": new_segment,
            "updated_at": run_timestamp
        })

    # Add new customers
    for i in range(NEW_CUSTOMERS):

        customer_id = create_id("CUST")
        first = random.choice(first_names)
        last = random.choice(last_names)

        customer_records.append({
            "customer_id": customer_id,
            "first_name": first,
            "last_name": last,
            "email": maybe_null(
                f"{first.lower()}.{last.lower()}@gmail.com"
            ),
            "phone": maybe_null(
                f"9{random.randint(100000000, 999999999)}"
            ),
            "city": random.choice(cities),
            "gender": random.choice(["Male", "Female", "Other"]),
            "registration_date": current_time.strftime("%Y-%m-%d"),
            "customer_segment": random.choice(
                ["Regular", "Premium", "VIP"]
            ),
            "updated_at": run_timestamp
        })

write_data(customer_records, CUSTOMER_PATH)


# ============================================================
# 5. GET CUSTOMER IDs FOR NEW ORDERS
# ============================================================

all_customer_df = read_existing(CUSTOMER_PATH)

customer_ids = [
    row["customer_id"]
    for row in all_customer_df.select("customer_id")
        .distinct()
        .collect()
]


# ============================================================
# 6. GENERATE UNIQUE ORDERS
# ============================================================

order_records = []

for i in range(NEW_ORDERS):

    order_id = create_id("ORD")

    product_id, product_name, price = random.choice(products)

    quantity = random.randint(1, 5)

    order_records.append({
        "order_id": order_id,
        "customer_id": random.choice(customer_ids),
        "product_id": product_id,
        "product_name": maybe_null(product_name, 0.02),
        "quantity": str(quantity),
        "unit_price": str(price),
        "total_amount": str(quantity * price),
        "payment_method": maybe_null(
            random.choice(payment_methods), 0.04
        ),
        "order_date": current_time.strftime("%Y-%m-%d"),
        "updated_at": run_timestamp
    })

write_data(order_records, ORDERS_PATH)


# ============================================================
# 7. GENERATE INITIAL STATUS FOR NEW ORDERS
# ============================================================

status_records = []

for order in order_records:

    status_records.append({
        "status_id": create_id("STS"),
        "order_id": order["order_id"],
        "status": "PLACED",
        "status_sequence": "1",
        "status_date": current_time.strftime("%Y-%m-%d %H:%M:%S"),
        "status_description": "Order placed successfully",
        "updated_at": run_timestamp
    })


# ============================================================
# 8. ADVANCE EXISTING ORDER STATUS
# ============================================================

existing_status_df = read_existing(STATUS_PATH)

if existing_status_df is not None:

    status_window = (
        Window.partitionBy("order_id")
        .orderBy(
            F.col("status_sequence").cast("int").desc(),
            F.col("status_date").desc()
        )
    )

    latest_status_df = (
        existing_status_df
        .withColumn("rn", F.row_number().over(status_window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # Only non-terminal orders can progress
    active_orders = (
        latest_status_df
        .filter(~F.col("status").isin(*terminal_statuses))
        .collect()
    )

    selected_orders = random.sample(
        active_orders,
        min(STATUS_UPDATES, len(active_orders))
    )

    for old in selected_orders:

        old_sequence = int(old["status_sequence"])

        # Never generate an event after DELIVERED
        if old["status"] in terminal_statuses:
            continue

        # Allow cancellation only before shipping
        if old["status"] in ["PLACED", "CONFIRMED"] and random.random() < 0.05:
            next_status = "CANCELLED"
            next_sequence = old_sequence + 1

        else:

            next_sequence = old_sequence + 1

            if next_sequence > len(status_hierarchy):
                continue

            next_status = status_hierarchy[next_sequence - 1]

        status_records.append({
            "status_id": create_id("STS"),
            "order_id": old["order_id"],
            "status": next_status,
            "status_sequence": str(next_sequence),
            "status_date": current_time.strftime("%Y-%m-%d %H:%M:%S"),
            "status_description": (
                f"Order {next_status.lower().replace('_', ' ')}"
            ),
            "updated_at": run_timestamp
        })


write_data(status_records, STATUS_PATH)


# ============================================================
# 9. EXECUTION SUMMARY
# ============================================================

print("\n========== DATA GENERATION SUMMARY ==========")

print("Customers generated:", len(customer_records))
print("New orders generated:", len(order_records))
print("Status events generated:", len(status_records))

print("\nCustomer path:", CUSTOMER_PATH)
print("Orders path:", ORDERS_PATH)
print("Status path:", STATUS_PATH)

print("\nDATA GENERATION COMPLETED SUCCESSFULLY")
